# Clause finder: plain-language answers from building regulations

Ask a question in everyday words. The tool finds the relevant clauses, explains them simply, and shows the original text, page and source so you can check it.

**How to use this notebook:** click the play button on each cell from the top, one at a time. Wait for the green tick before running the next. Cells marked **YOUR TURN** need something from you.

Research aid only. Not legal or engineering advice.

## Step 1: install the tools (2-3 minutes)

In [ ]:
!pip install -q pypdf rank_bm25 sentence-transformers gradio requests

## Step 2: create the project files
The next cells write the project's code into this notebook's workspace. Just run them. You do not need to read the code yet.

In [ ]:
%%writefile ingest.py
"""Step 1: turn a regulations PDF (or .txt) into a list of clauses.

Usage:
    python ingest.py data/part2.pdf                          # writes clauses.jsonl
    python ingest.py data/part2.pdf --chapters 6-11          # keep only chapters 6 to 11
    python ingest.py data/a.pdf data/b.pdf --out clauses.jsonl

Each record: id, source, clause_no, title, text, page, page_end, part.
  page     = page where THIS piece of text starts
  page_end = page where it ends (differs from page when a long clause runs across pages)

A "clause" starts at a line that begins with a number like 6.2 or 6.2.1 followed by a heading.

Things this script deliberately ignores:
  - contents pages and contents-style lines (they end in a page number)
  - running page headers and footers
  - table rows and stray numbers (a clause heading must start with a word)
  - with --chapters, numbered lines whose first number is outside the chapters you list
Long clauses are cut at sentence or word boundaries, never in the middle of a word.
"""
import argparse
import json
import re
import sys
from collections import Counter
from pathlib import Path

CLAUSE_RE = re.compile(r"^\s*(\d{1,2}(?:\.\d{1,3}){1,5})\.?\s+(\S.*)$")
LEADER_RE = re.compile(r"(\.\s?){4,}")
PAGEREF_RE = re.compile(r"\s\d{1,3}\s*$")  # a line that ends in a bare page number
FOOTER_RE = re.compile(
    r"Comprehensive (General )?Development Control Regulations|UD\s*&\s*UHD|^\s*Page\s+\d+",
    re.I,
)
MAX_CHARS = 1200   # target size of a searchable piece
TOC_MAX_PAGE = 10  # contents pages are only looked for in the first pages


def parse_chapters(spec):
    """'6-11' or '6,7,8' or '6-8,10' -> {6, 7, ...}. Empty/None -> None (no filter)."""
    if not spec:
        return None
    out = set()
    for part in spec.split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-", 1)
            out.update(range(int(a), int(b) + 1))
        elif part:
            out.add(int(part))
    return out or None


UNIT_START_RE = re.compile(r"^(mtr|mtrs|mts|mt|mtr\.|m|sq|sqm|sq\.mt|metre|metres|meter|meters|ft|feet|nos|no|%)\b", re.I)


def valid_number(num):
    """Real clause numbers never contain a zero component (6.00, 10.00 are table values like '6.00 mtr.')."""
    return all(int(p) > 0 for p in num.split("."))


def valid_heading(rest):
    """A real clause heading starts with a word, not with table numbers, units or symbols."""
    rest = rest.strip()
    if not rest or not (rest[0].isalpha() or rest[0] in "\"'(“"):
        return False
    if UNIT_START_RE.match(rest):
        return False
    letters = sum(c.isalpha() for c in rest)
    non_space = sum(not c.isspace() for c in rest) or 1
    return letters >= 6 and letters / non_space >= 0.6


def is_toc_page(lines):
    """Early page where many lines end in a page number = a table-of-contents page."""
    nonempty = [l for l in lines if l.strip()]
    if len(nonempty) < 8:
        return False
    hits = sum(bool(PAGEREF_RE.search(l)) for l in nonempty)
    return hits >= 8 and hits / len(nonempty) >= 0.4


def read_pages(path: Path):
    """Return a list of (page_number, text)."""
    if path.suffix.lower() == ".txt":
        return [(1, path.read_text(encoding="utf-8", errors="ignore"))]
    from pypdf import PdfReader

    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        pages.append((i, page.extract_text() or ""))
    return pages


def split_clauses(pages, source, chapters=None, stats=None):
    stats = stats if stats is not None else {}
    clauses = []
    cur = None
    for page_no, text in pages:
        lines = text.splitlines()
        if page_no <= TOC_MAX_PAGE and is_toc_page(lines):
            stats["toc_pages"] = stats.get("toc_pages", 0) + 1
            continue
        for line in lines:
            line = " ".join(line.split())
            if not line or FOOTER_RE.search(line) or LEADER_RE.search(line):
                continue
            m = CLAUSE_RE.match(line)
            starts_clause = (
                m
                and valid_number(m.group(1))
                and valid_heading(m.group(2))
                and (chapters is None or int(m.group(1).split(".")[0]) in chapters)
                and not (page_no <= TOC_MAX_PAGE and PAGEREF_RE.search(line))  # contents-style line
            )
            if starts_clause:
                if cur:
                    clauses.append(cur)
                cur = {
                    "source": source,
                    "clause_no": m.group(1),
                    "title": m.group(2)[:120],
                    "text": m.group(2),
                    "marks": [(0, page_no)],  # (character offset, page) pairs
                }
            elif cur:
                if cur["marks"][-1][1] != page_no:
                    cur["marks"].append((len(cur["text"]) + 1, page_no))
                cur["text"] += " " + line
    if cur:
        clauses.append(cur)
    return clauses


def split_text(text, limit=MAX_CHARS):
    """Cut text into pieces of about `limit` characters, at sentence or word boundaries.
    Returns a list of (start_offset, piece)."""
    pieces, start, n = [], 0, len(text)
    while start < n:
        end = min(start + limit, n)
        if end < n:
            window = text[start:end]
            cut = max(window.rfind(". "), window.rfind("; "), window.rfind(": "), window.rfind(") "))
            if cut < limit * 0.5:
                cut = window.rfind(" ")
            if cut <= 0:
                cut = len(window) - 1
            end = start + cut + 1
        piece = text[start:end].strip()
        if piece:
            pieces.append((start, piece))
        start = end
        while start < n and text[start] == " ":
            start += 1
    return pieces


def page_at(marks, offset):
    page = marks[0][1]
    for off, pg in marks:
        if off <= offset:
            page = pg
        else:
            break
    return page


def chunk_long(clauses):
    out = []
    for c in clauses:
        text = c["text"].strip()
        if len(text) < 25:  # headings with no body, stray lines
            continue
        pieces = split_text(text)
        for j, (start, piece) in enumerate(pieces):
            out.append({
                "source": c["source"], "clause_no": c["clause_no"], "title": c["title"],
                "text": piece,
                "page": page_at(c["marks"], start),
                "page_end": page_at(c["marks"], start + len(piece) - 1),
                "part": j + 1,
            })
    for i, c in enumerate(out):
        c["id"] = i
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("files", nargs="+")
    ap.add_argument("--out", default="clauses.jsonl")
    ap.add_argument("--chapters", default=None, help="keep only these chapters, e.g. 6-11")
    args = ap.parse_args()
    chapters = parse_chapters(args.chapters)

    all_clauses = []
    for f in args.files:
        p = Path(f)
        if not p.exists():
            sys.exit(f"File not found: {p}")
        pages = read_pages(p)
        stats = {}
        found = split_clauses(pages, p.name, chapters, stats)
        print(f"{p.name}: {len(pages)} pages, {len(found)} clauses found, "
              f"{stats.get('toc_pages', 0)} contents page(s) skipped")
        all_clauses.extend(found)

    chunks = chunk_long(all_clauses)
    with open(args.out, "w", encoding="utf-8") as fh:
        for c in chunks:
            fh.write(json.dumps(c, ensure_ascii=False) + "\n")
    print(f"Wrote {len(chunks)} searchable chunks to {args.out}")

    by_chapter = Counter(c["clause_no"].split(".")[0] for c in chunks)
    summary = ", ".join(f"ch {k}: {v}" for k, v in sorted(by_chapter.items(), key=lambda kv: int(kv[0])))
    print("Chunks per chapter ->", summary)

    if len(chunks) < 20:
        print("Warning: very few clauses found. Is the PDF scanned (images)? "
              "Open it and check you can select text.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile concepts.py
"""Query understanding: turn everyday wording into the terms regulations actually use.

IMPORTANT: this map is HAND-WRITTEN. Every entry must be checked against the real document
before you rely on it. "definition" is a general explanation, NOT text from the regulation,
and the app labels it that way.

What this module does:
  1. normalize()      - fixes likely spelling mistakes by snapping words to known vocabulary
  2. detect_concepts  - finds which regulatory concepts the question is about
  3. analyze_query    - returns concepts, extra search terms, and details that are missing
"""
import difflib
import re

CONCEPTS = [
    {
        "name": "Setback / margin (open space around the building)",
        "triggers": ["setback", "set back", "margin", "empty space", "open space", "space between",
                     "leave space", "how close can i", "how far must", "how far should", "how far from my",
                     "distance from my", "distance from the", "edge of my land",
                     "edge of my plot", "edge of the plot", "edge of land", "right up to",
                     "boundary", "neighbour", "neighbor", "side space", "back yard", "front yard",
                     "clearance"],
        "terms": ["margin", "setback", "set back", "side margin", "rear margin",
                  "road side margin", "open to sky"],
        "definition": "The open space that must be left between a building and the boundaries of its plot (front, sides and rear).",
        "factors": ["plot size", "road width", "building height", "building use"],
    },
    {
        "name": "Ground coverage",
        "triggers": ["ground coverage", "plot coverage", "how much of my land", "how much of the plot",
                     "cover with", "covered by", "land covered", "footprint", "built on the plot",
                     "how much can i build on"],
        "terms": ["ground coverage", "plot coverage", "built up area", "coverage"],
        "definition": "The share of the plot area that the building's footprint is allowed to cover.",
        "factors": ["plot size", "building use"],
    },
    {
        "name": "Floor space index (FSI)",
        "triggers": ["fsi", "floor space index", "floor area ratio", "total built up", "how much area can i build",
                     "construction area allowed", "how much can i build in total", "built up area"],
        "terms": ["floor space index", "FSI", "chargeable FSI", "built up area"],
        "definition": "The ratio of the total floor area a building may have to the area of its plot.",
        "factors": ["plot size", "road width", "building use"],
    },
    {
        "name": "Building height",
        "triggers": ["another floor", "extra floor", "add a floor", "add floor", "more floors", "how many floors",
                     "how tall", "how high", "height", "storey", "storeys", "story", "stories", "upper floor"],
        "terms": ["building height", "permissible height", "height limit", "number of storeys"],
        "definition": "The maximum height or number of storeys a building may have.",
        "factors": ["road width", "plot size", "building use"],
    },
    {
        "name": "Parking",
        "triggers": ["parking", "park my car", "car space", "car spot", "vehicles", "garage", "parking space"],
        "terms": ["parking", "parking space", "parking requirement", "basement parking"],
        "definition": "The number and size of vehicle parking spaces a building must provide.",
        "factors": ["building use", "plot size"],
    },
    {
        "name": "Permitted land use (zoning)",
        "triggers": ["open a shop", "run a shop", "start a business", "shop at home", "office at home",
                     "factory", "commercial", "use my land", "use the land", "permitted use", "allowed to run",
                     "business from home", "zone", "zoning"],
        "terms": ["permissible use", "land use", "use zone", "residential zone", "commercial use", "permitted uses"],
        "definition": "Which activities (housing, shops, offices, industry and so on) are allowed on land in a given zone.",
        "factors": ["building use", "city or authority"],
    },
    {
        "name": "Basement",
        "triggers": ["basement", "cellar", "underground floor", "underground parking"],
        "terms": ["basement", "podium", "basement parking"],
        "definition": "A floor built partly or wholly below ground level.",
        "factors": ["plot size", "building use"],
    },
    {
        "name": "Road width and building line",
        "triggers": ["road widening", "building line", "road width", "how close to the road", "near the road",
                     "next to the road", "from the road", "to the road", "close to the road",
                     "road boundary", "road side"],
        "terms": ["road width", "building line", "road boundary", "road widening", "road side margin"],
        "definition": "Rules that depend on the width of the road in front of a plot, and the line beyond which building is not allowed.",
        "factors": ["road width", "plot size"],
    },
    {
        "name": "Fire safety, stairs and lifts",
        "triggers": ["fire exit", "fire escape", "fire safety", "lift", "elevator", "staircase", "stairs",
                     "stairway", "escape route", "emergency exit"],
        "terms": ["fire escape", "fire safety", "staircase", "lift", "means of escape"],
        "definition": "Requirements for safe exit, stairs and lifts in buildings.",
        "factors": ["building height", "building use"],
    },
    {
        "name": "Rainwater harvesting",
        "triggers": ["rain water", "rainwater", "water harvesting", "groundwater", "collect rain"],
        "terms": ["rainwater harvesting", "recharge", "groundwater recharge"],
        "definition": "Collecting and storing rainwater, or letting it soak into the ground.",
        "factors": ["plot size"],
    },
    {
        "name": "Permission and approval",
        "triggers": ["permission", "approval", "approve", "permit", "sanction", "licence", "license",
                     "building plan approval", "how long to approve"],
        "terms": ["development permission", "building permission", "application", "sanction"],
        "definition": "The process and conditions for getting official permission to build.",
        "factors": ["city or authority"],
    },
]

FACTOR_PATTERNS = {
    "plot size": r"\d+\s*(sq|square|m2|sqm|sq\.?\s?m|yard|gaj)|plot size|plot area|square",
    "road width": r"road.*\d+|\d+\s*(m|metre|meter|ft|feet)\b.*road|road width|wide road|narrow road",
    "building height": r"\d+\s*(storey|story|floor|m\b|metre|meter)|height|storeys|floors",
    "building use": r"residential|commercial|industrial|shop|office|school|house|flat|apartment|factory|hospital|home",
    "city or authority": r"ahmedabad|surat|rajkot|vadodara|gandhinagar|auda|suda|city|municipal|authority",
}

_WORD = re.compile(r"[a-z]+")
_VOCAB = sorted({w for c in CONCEPTS for t in c["triggers"] for w in _WORD.findall(t) if len(w) >= 5})


def normalize(query):
    """Lowercase, keep letters only, and snap near-miss spellings to known vocabulary."""
    out = []
    for w in _WORD.findall(query.lower()):
        if len(w) >= 5 and w not in _VOCAB:
            match = difflib.get_close_matches(w, _VOCAB, n=1, cutoff=0.84)
            if match:
                w = match[0]
        out.append(w)
    return " ".join(out)


def detect_concepts(query):
    norm = " " + normalize(query) + " "
    found = []
    for c in CONCEPTS:
        if any(" " + t + " " in norm for t in c["triggers"]):
            found.append(c)
    return found


def analyze_query(query):
    found = detect_concepts(query)
    terms, factors = [], []
    for c in found:
        terms += [t for t in c["terms"] if t not in terms]
        factors += [f for f in c["factors"] if f not in factors]
    low = query.lower()
    missing = [f for f in factors if not re.search(FACTOR_PATTERNS[f], low)]
    return {
        "concepts": [c["name"] for c in found],
        "definitions": [(c["name"], c["definition"]) for c in found],
        "terms": terms,
        "expanded": (query + " " + " ".join(terms)).strip(),
        "missing_details": missing,
    }


if __name__ == "__main__":
    import sys

    q = " ".join(sys.argv[1:]) or "Can I build my house right up to the edge of my land?"
    print(analyze_query(q))

In [ ]:
%%writefile search.py
"""The search engine. Modes:

  keyword  - BM25 word matching (fast, exact, but misses different wording)
  semantic - embeddings (matches meaning, can be loosely related)
  hybrid   - both combined with Reciprocal Rank Fusion (RRF)

On top of that, expand=True adds regulatory terms found by concepts.py to the query
(for example "edge of my land" also searches for "margin" and "setback").

Embedder choice:
  "minilm" - sentence-transformers all-MiniLM-L6-v2 (real semantic model; needs internet once to download)
  "lsa"    - TF-IDF + SVD fallback (runs anywhere, weaker; used if minilm is unavailable)

assess_evidence() decides whether the retrieved clauses are relevant enough to answer from.
Its thresholds are PROVISIONAL: recalibrate them with the diagnostics printed by eval.py.
"""
import json
import re

import numpy as np
from rank_bm25 import BM25Okapi

from concepts import analyze_query

STOP = set("the a an of to in and or for is are be on at by with as that this it from any shall".split())
GENERIC = set("what how can does need much many have must get who when where which about there their "
              "plot building house land rule rules regulation regulations required allowed".split())

MIN_SEM = 0.30                # no topic recognised: best semantic score must reach this
MIN_SEM_WITH_CONCEPT = 0.20   # topic recognised: a lower bar is enough
MIN_OVERLAP = 0.25            # no topic recognised: share of question words found in top clauses


def stem(t):
    """Very light plural stemming so 'margins' matches 'margin' and 'buildings' matches 'building'."""
    if len(t) > 4 and t.endswith("ies"):
        return t[:-3] + "y"
    if len(t) > 4 and t.endswith("sses"):
        return t[:-2]
    if len(t) > 3 and t.endswith("s") and not t.endswith(("ss", "us", "is")):
        return t[:-1]
    return t


def tokenize(text):
    return [stem(t) for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOP]


def load_clauses(path="clauses.jsonl"):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


class Searcher:
    def __init__(self, clauses, embedder="auto"):
        self.clauses = clauses
        self.docs = [c["title"] + ". " + c["text"] for c in clauses]
        self.bm25 = BM25Okapi([tokenize(c["title"] + " " + d) for c, d in zip(clauses, self.docs)])
        self.title_bm25 = BM25Okapi([tokenize(c["title"]) or ["_"] for c in clauses])
        self.embedder_name, self._embed, self.doc_vecs = self._build_embedder(embedder)

    # ---------- embeddings ----------
    def _build_embedder(self, choice):
        if choice in ("auto", "minilm"):
            try:
                from sentence_transformers import SentenceTransformer

                model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
                vecs = model.encode(self.docs, normalize_embeddings=True, show_progress_bar=False)
                return "minilm", (lambda q: model.encode([q], normalize_embeddings=True)[0]), np.asarray(vecs)
            except Exception as e:  # no package / no internet
                if choice == "minilm":
                    raise
                print(f"[search] MiniLM unavailable ({type(e).__name__}); using LSA fallback.")
        from sklearn.decomposition import TruncatedSVD
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.preprocessing import normalize

        tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None, sublinear_tf=True)
        X = tfidf.fit_transform(self.docs)
        k = max(2, min(128, X.shape[0] - 1, X.shape[1] - 1))
        svd = TruncatedSVD(n_components=k, random_state=0)
        vecs = normalize(svd.fit_transform(X))
        return "lsa", (lambda q: normalize(svd.transform(tfidf.transform([q])))[0]), vecs

    # ---------- query understanding ----------
    def analyze(self, query):
        return analyze_query(query)

    # ---------- search ----------
    def search(self, query, mode="hybrid", k=5, expand=True, titles=None, dedupe=True):
        """Return the top-k results.
        expand : add regulatory terms from the term map to the query
        titles : also rank by clause HEADING match (hybrid mode only). Defaults to the value of expand.
        dedupe : show each clause once (its best passage); other matching passages of the same
                 clause are merged into result['context'] so nothing is silently dropped."""
        titles = expand if titles is None else titles
        qx = analyze_query(query)["expanded"] if expand else query
        bm = self.bm25.get_scores(tokenize(qx))
        sem = self.doc_vecs @ self._embed(query)  # meaning search uses the ORIGINAL question; the term list would dilute it

        if mode == "keyword":
            order = list(np.argsort(-bm))
            scores = bm
        elif mode == "semantic":
            order = list(np.argsort(-sem))
            scores = sem
        else:  # hybrid: Reciprocal Rank Fusion of the rankings
            rankings = [list(np.argsort(-bm)), list(np.argsort(-sem))]
            if titles:
                tq = [t for t in tokenize(qx) if t not in GENERIC and len(t) > 2]
                tit = self.title_bm25.get_scores(tq) if tq else np.zeros(len(self.clauses))
                if tit.max() > 0:
                    rankings.append([i for i in np.argsort(-tit)[:10] if tit[i] > 0])
            fused = {}
            for n, ranking in enumerate(rankings):
                weight = 0.5 if n == 2 else 1.0  # heading match is a weaker vote than keyword/meaning
                for rank, idx in enumerate(ranking[:50]):
                    fused[idx] = fused.get(idx, 0.0) + weight / (60 + rank)
            order = sorted(fused, key=fused.get, reverse=True)
            scores = np.zeros(len(self.clauses))
            for i, s in fused.items():
                scores[i] = s

        if not dedupe:
            return [
                {**self.clauses[i], "context": self.clauses[i]["text"], "n_passages": 1, "score": float(scores[i]),
                 "sem_score": float(sem[i]), "bm25_score": float(bm[i])}
                for i in order[:k]
            ]
        return self._group_by_clause(order, scores, sem, bm, k)

    def _group_by_clause(self, order, scores, sem, bm, k):
        groups, ranked = {}, []
        for i in order[: max(80, k * 3)]:
            c = self.clauses[i]
            key = (c["source"], c["clause_no"])
            if key in groups:
                if len(groups[key]["sib"]) < 2:
                    groups[key]["sib"].append(i)
                continue
            if len(ranked) >= k:
                continue  # only collecting extra passages for clauses already chosen
            groups[key] = {"i": i, "sib": []}
            ranked.append(groups[key])
        out = []
        for g in ranked:
            i = g["i"]
            parts = sorted([self.clauses[i]] + [self.clauses[j] for j in g["sib"]], key=lambda c: c.get("part", 1))
            context, prev = "", None
            for c in parts:
                if prev is not None:
                    context += " " if c.get("part", 1) == prev + 1 else " … "
                context += c["text"]
                prev = c.get("part", 1)
            out.append({
                **self.clauses[i],
                "context": context,
                "n_passages": len(parts),
                "page": min(c["page"] for c in parts),
                "page_end": max(c.get("page_end", c["page"]) for c in parts),
                "score": float(scores[i]), "sem_score": float(sem[i]), "bm25_score": float(bm[i]),
            })
        return out


def assess_evidence(query, info, results, min_sem=MIN_SEM, min_sem_concept=MIN_SEM_WITH_CONCEPT,
                    min_overlap=MIN_OVERLAP):
    """Return (status, reason). status is 'ok' or 'none'. Never proves a clause APPLIES; it only
    says whether the retrieved text is related enough to the question to be worth answering from."""
    if not results:
        return "none", "No clauses were retrieved."
    top_sem = max(r["sem_score"] for r in results)
    if info["concepts"]:
        if top_sem >= min_sem_concept:
            return "ok", "Topic recognised and related clauses found."
        return "none", f"Topic recognised but the closest clauses are only weakly related (score {top_sem:.2f})."
    words = [t for t in tokenize(query) if len(t) > 2 and t not in GENERIC]
    blob = " ".join(r["title"] + " " + r["text"] for r in results[:3]).lower()
    overlap = sum(w in blob for w in words) / max(len(words), 1)
    if top_sem >= min_sem and overlap >= min_overlap:
        return "ok", "Related clauses found."
    return "none", f"No recognised topic, and the closest clauses are weakly related (score {top_sem:.2f}, word overlap {overlap:.0%})."


if __name__ == "__main__":
    import sys

    s = Searcher(load_clauses())
    q = " ".join(sys.argv[1:]) or "front margin for a residential plot"
    print(f"Embedder: {s.embedder_name}")
    info = s.analyze(q)
    print("Topics:", info["concepts"], "| added terms:", info["terms"])
    res = s.search(q)
    for r in res:
        print(f"{r['clause_no']:<10} p.{r['page']:<4} sem={r['sem_score']:.2f}  {r['title'][:70]}")
    print("Evidence check:", assess_evidence(q, info, res))

In [ ]:
%%writefile answer.py
"""Plain-language answers that are grounded in the retrieved clauses and CHECKED afterwards.

Pipeline:
  1. understand the question (concepts.py) and retrieve clauses (search.py)
  2. if the evidence looks too weak -> refuse (status "insufficient")
  3. if no AI model is available    -> show clauses only (status "evidence_only")
  4. otherwise ask the model for a JSON answer using ONLY the retrieved clauses
  5. VERIFY the answer in code. If any check fails, the AI text is withheld (status "unverified")

The checks (verify_answer) are what stop a confident wrong number from reaching the user:
  - every cited clause number must be one of the retrieved clauses
  - every number in the answer must appear in the cited clauses' text
  - every spelled-out number (two, three...) must appear in the cited clauses' text
  - an answer with no citations is rejected unless the model said "insufficient"

These checks reduce risk. They do NOT prove the answer is correct or that a clause applies to
a particular plot. The interface says so.
"""
import json
import os
import re
import time

import requests

from search import assess_evidence

NUM_RE = re.compile(r"\d+(?:\.\d+)*")
NUMBER_WORDS = ("one two three four five six seven eight nine ten eleven twelve fifteen twenty thirty "
                "fifty hundred thousand half double triple").split()

PROMPT = """You explain building regulations to a person with no technical background.
Use ONLY the numbered clauses below. Do not use outside knowledge. Do not guess.

QUESTION: {question}

CLAUSES:
{clauses}

First decide: does any clause DIRECTLY answer the subject of the question? A clause that is only about a related topic
(for example a clause about heritage areas, internal roads or chawls when the question is about an ordinary house) does NOT
count. If no clause directly answers it, set "insufficient" to true, leave the other fields empty, and stop.

Ignore clauses that apply only to a special case (basement, heritage core area, fuelling or electric vehicle station, land-locked
plot, industrial or commercial use, redevelopment, core walled city) unless the question itself mentions that case. If only
special-case clauses are available, set "insufficient" to true.

If a clause does answer it:
- "evidence": one to three entries. Each has the clause number and a "quote" copied WORD FOR WORD from that clause (at most
  30 words) that directly answers the question. Do not change, shorten inside, or paraphrase the quote.
- "direct_answer": ONE or TWO plain sentences. State the actual rule first (what is required, allowed or not allowed), using
  what the clause says. Only after that, say what changes the exact figure. Do NOT start with "It depends" unless the clauses
  really give no rule that applies in every case; even then say first what the clauses require in all cases.
- "simple_explanation": three to five short sentences in everyday words saying what this means for the person.
- "by_clause": one entry for EACH clause you used. "says" explains that clause in simple words, one to three sentences. Give
  a number only if it is written in that same clause's text.
- If a clause gives its values in a table, say what the table is about and which factors choose the row (plot area, road
  width, building height, use) and tell the user to read that table for the exact figure. Tables are flattened, so you cannot
  tell which number belongs to which row: do NOT pick a value from a table. Numbers in ordinary sentences can be stated.
- "check_next": short list of facts the person should find out to get an exact answer (for example "your plot area in sq.mt").
- Never state a number, measurement or percentage unless it appears in the clause you attribute it to.

Return ONLY JSON with these keys:
{{"insufficient": false,
  "evidence": [{{"clause": "6.7.2", "quote": "exact words from the clause"}}],
  "direct_answer": "...",
  "simple_explanation": "...",
  "by_clause": [{{"clause": "6.7.2", "says": "..."}}],
  "check_next": ["..."]}}"""


JUDGE_PROMPT = """You are a strict reviewer of an answer about building regulations.

QUESTION: {question}

TEXT OF THE CLAUSES THE ANSWER RELIES ON (from the regulation):
{quotes}

DRAFT ANSWER: {answer}

Check two things, using ONLY the clause text above:
1. "supported": the statements in the draft answer follow from the clause text above (it may explain the clause in simpler
   words; it must not add rules or figures that the text does not contain).
2. "applies": the clause text is about the SAME situation the question asks about. Answer false if the quotes are about a
   different subject (for example a different structure than the one asked about, such as a compound wall versus a partition
   wall), a special case (heritage area, fuelling station, land-locked plot, redevelopment) when the question is about an
   ordinary house or plot, or a different quantity or unit than the question mentions.
Return ONLY JSON: {{"supported": true, "applies": true, "reason": "one short sentence"}}"""


# ---------------------------------------------------------------- model access
def list_gemini_models(api_key):
    r = requests.get("https://generativelanguage.googleapis.com/v1beta/models",
                     params={"key": api_key, "pageSize": 200}, timeout=30)
    r.raise_for_status()
    return [m["name"].split("/")[-1] for m in r.json().get("models", [])
            if "generateContent" in m.get("supportedGenerationMethods", [])]


def pick_gemini_model(api_key):
    """Choose a current general-purpose Flash model from what this key can actually use."""
    if os.environ.get("GEMINI_MODEL"):
        return os.environ["GEMINI_MODEL"]
    names = list_gemini_models(api_key)
    bad = ("lite", "preview", "exp", "image", "tts", "live", "thinking", "audio", "embedding", "vision", "latest")
    flash = sorted([n for n in names if "flash" in n and not any(b in n for b in bad)], reverse=True)
    if flash:
        return flash[0]
    raise RuntimeError("No Flash model found for this key. Models available: " + ", ".join(names[:15]))


def make_gemini_llm(api_key):
    """Return a function prompt -> text, or raise a clear error."""
    model = pick_gemini_model(api_key)

    def llm(prompt):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
        body = {"contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"temperature": 0, "responseMimeType": "application/json"}}
        last = None
        for attempt in range(int(os.environ.get("GEMINI_RETRIES", "5"))):  # retry timeouts and temporary overload (429/500/503)
            try:
                r = requests.post(url, params={"key": api_key}, json=body, timeout=int(os.environ.get("GEMINI_TIMEOUT", "150")))
            except requests.exceptions.RequestException as e:
                last = f"{type(e).__name__}: {e}"
            else:
                if r.status_code == 200:
                    return r.json()["candidates"][0]["content"]["parts"][0]["text"]
                last = f"Gemini API error {r.status_code}: {r.text[:300]}"
                if r.status_code not in (429, 500, 503):
                    break
            time.sleep(min(8 * (2 ** attempt), 60))
        raise RuntimeError(last)

    llm.model_name = model
    return llm


# ---------------------------------------------------------------- verification
def _norm_num(n):
    n = n.replace(",", "")
    try:
        return "%g" % float(n)
    except ValueError:
        return n


def parse_json(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    return json.loads(text)


def cited_clauses(answer):
    """Clause numbers the answer relies on: by_clause entries plus any legacy 'cited' list."""
    out = [str(e.get("clause", "")).strip() for e in answer.get("by_clause", []) if isinstance(e, dict)]
    out += [str(e.get("clause", "")).strip() for e in answer.get("evidence", []) if isinstance(e, dict)]
    out += [str(x).strip() for x in answer.get("cited", [])]
    return [c for i, c in enumerate(out) if c and c not in out[:i]]


def _number_problems(text, allowed, evidence_low, label):
    problems = []
    for n in NUM_RE.findall(str(text).replace(",", "")):
        if _norm_num(n) not in allowed:
            problems.append(f"{label} mentions the number {n}, which is not in the cited clause text")
    for w in NUMBER_WORDS:
        if re.search(rf"\b{w}\b", str(text).lower()) and not re.search(rf"\b{w}\b", evidence_low):
            problems.append(f"{label} mentions '{w}', which is not in the cited clause text")
    return problems


def verify_answer(answer, clauses):
    """Return a list of problems. Empty list = all checks passed."""
    problems = []
    by_no = {}
    for c in clauses:
        by_no.setdefault(c["clause_no"], []).append(c.get("context", c["text"])[:1800])
    cited = cited_clauses(answer)

    unknown = [c for c in cited if c not in by_no]
    if unknown:
        problems.append("cites clauses that were not retrieved: " + ", ".join(unknown))
    if not cited and not answer.get("insufficient"):
        problems.append("gives an answer without citing any clause")

    def allowed_for(nos):
        ev = " ".join(" ".join(by_no[c]) for c in nos if c in by_no)
        nums = {_norm_num(n) for n in NUM_RE.findall(ev.replace(",", ""))}
        nums |= {_norm_num(x) for x in nos} | {_norm_num(x) for x in by_no}
        return nums, ev.lower()

    # every answer must carry at least one quote that really appears, word for word, in the clause it names
    def _norm(t):
        return re.sub(r"[^a-z0-9.%]+", " ", str(t).lower()).strip()

    good_quotes = 0
    for e in answer.get("evidence", []):
        if not isinstance(e, dict):
            continue
        no, quote = str(e.get("clause", "")).strip(), _norm(e.get("quote", ""))
        text = _norm(" ".join(c.get("context", c["text"]) for c in clauses if c["clause_no"] == no))
        if len(quote.split()) >= 4 and quote in text:
            good_quotes += 1
        else:
            problems.append(f"quote from clause {no} does not appear in that clause")
    if not answer.get("insufficient") and good_quotes == 0:
        problems.append("no word-for-word quote from a retrieved clause supports the answer")

    # overall fields may use numbers from any cited clause
    allowed_all, ev_all = allowed_for(cited)
    for label, f in [("the direct answer", answer.get("direct_answer", "")),
                     ("the explanation", answer.get("simple_explanation", ""))] + \
                    [("a 'check next' item", x) for x in answer.get("check_next", [])] + \
                    [("a condition", x) for x in answer.get("conditions", [])]:
        problems += _number_problems(f, allowed_all, ev_all, label)

    # each per-clause explanation may only use numbers from ITS OWN clause
    for e in answer.get("by_clause", []):
        if not isinstance(e, dict):
            continue
        no = str(e.get("clause", "")).strip()
        allowed_one, ev_one = allowed_for([no])
        problems += _number_problems(e.get("says", ""), allowed_one, ev_one, f"the explanation of clause {no}")
    return sorted(set(problems))


# ---------------------------------------------------------------- main entry
def judge_answer(llm, question, answer, results=()):
    """Second, independent model call that sees only the question, the verified quotes and the draft answer.
    Returns (ok, reason). A failed call counts as NOT ok: we prefer withholding over showing an unchecked answer."""
    nos = set(cited_clauses(answer))
    quotes = "\n".join(f"[{r['clause_no']}] {r['title']}: {r.get('context', r['text'])[:1800]}"
                       for r in results if r["clause_no"] in nos)
    try:
        j = parse_json(llm(JUDGE_PROMPT.format(question=question, quotes=quotes,
                                               answer=answer.get("direct_answer", ""))))
    except Exception as e:
        return False, f"relevance check could not run ({type(e).__name__})"
    if j.get("supported") is True and j.get("applies") is True:
        return True, ""
    return False, str(j.get("reason", "the quotes do not answer this question"))[:200]


def answer_question(searcher, question, llm=None, k=5, expand=True, judge=True):  # term map + down-weighted heading match; chosen using 5 independent real questions (see README)
    info = searcher.analyze(question)
    results = searcher.search(question, mode="hybrid", k=k, expand=expand)
    status, reason = assess_evidence(question, info, results)
    out = {"question": question, "info": info, "results": results, "reason": reason,
           "answer": None, "problems": [], "status": None, "model": None}

    if status == "none":
        out["status"] = "insufficient"
        return out
    if llm is None:
        out["status"] = "evidence_only"
        return out

    clause_text = "\n".join(
        f"[{r['clause_no']}] (page {r['page']}) {r['title']}: {r.get('context', r['text'])[:1800]}" for r in results)
    try:
        raw = llm(PROMPT.format(question=question, clauses=clause_text))
        answer = parse_json(raw)
        if isinstance(answer, list) and len(answer) == 1:  # model sometimes wraps the object in a list
            answer = answer[0]
        if not isinstance(answer, dict):
            raise ValueError("model did not return a JSON object")
    except Exception as e:  # network, quota, bad JSON
        out["status"] = "llm_error"
        out["problems"] = [f"{type(e).__name__}: {e}"]
        return out

    out["model"] = getattr(llm, "model_name", None)
    if answer.get("insufficient"):
        out["status"] = "insufficient"
        out["reason"] = "The retrieved clauses do not clearly answer this question."
        return out

    problems = verify_answer(answer, results)
    if not problems and judge:
        ok, why = judge_answer(llm, question, answer, results)
        if not ok:
            out["status"] = "insufficient"
            out["reason"] = "The clauses found do not clearly answer this exact question (" + why + ")"
            return out
    out["answer"] = answer
    out["problems"] = problems
    out["status"] = "unverified" if problems else "answered"
    return out

In [ ]:
%%writefile render.py
"""Turn an answer_question() result into Markdown that keeps original text and AI text clearly apart."""
import json
import os
import re

DISCLAIMER = ("Research aid only. Not legal, engineering or official advice. Check the official, "
              "current notification and your local authority before relying on any clause.")


def _load_sources(path="sources.json"):
    if os.path.exists(path):
        try:
            return json.load(open(path, encoding="utf-8"))
        except Exception:
            return []
    return []


def source_info(filename, sources):
    low = filename.lower()
    for s in sources:
        if any(m in low for m in s.get("match", [])):
            return s
    return None


def to_markdown(res, sources_path="sources.json"):
    sources = _load_sources(sources_path)
    info, status = res["info"], res["status"]
    md = []

    if info["concepts"]:
        md.append("### What the system understood")
        for name, definition in info["definitions"]:
            md.append(f"- **Topic:** {name}\n  - *General explanation (not from the regulation):* {definition}")
        if info["terms"]:
            md.append(f"- **Regulatory terms searched as well:** {', '.join(info['terms'])}")
        if info["missing_details"]:
            md.append("- **Details that often change the answer, not given in your question:** "
                      + ", ".join(info["missing_details"]) + ". Check the clauses below for how these matter.")
        md.append("")

    if status == "answered":
        a = res["answer"]
        md.append("### Plain-language answer  \n*AI-generated from the clauses below. It is not the regulation. Check it against the original text.*")
        md.append(f"**{a.get('direct_answer', '')}**")
        md.append(a.get("simple_explanation", ""))
        if a.get("evidence"):
            md.append("**The regulation says (word for word, checked against the clause text):**")
            md += [f"> “{e.get('quote', '')}”  (clause {e.get('clause')})" for e in a["evidence"] if isinstance(e, dict)]
        if a.get("by_clause"):
            md.append("**What each clause says, in simple words:**")
            md += [f"- **Clause {e.get('clause')}:** {e.get('says', '')}" for e in a["by_clause"] if isinstance(e, dict)]
        if a.get("check_next"):
            md.append("**To get an exact answer, find out:**")
            md += [f"- {c}" for c in a["check_next"]]
        if a.get("conditions"):
            md.append("**Conditions and exceptions stated in the clauses:**")
            md += [f"- {c}" for c in a["conditions"]]
        from answer import cited_clauses
        md.append(f"*Clauses used: {', '.join(cited_clauses(a))}. Automatic checks passed: cited clauses exist and every number appears in the clause it is attributed to. This does not prove the clause applies to your situation.*")
    elif status == "unverified":
        md.append("### Plain-language answer withheld")
        md.append("The AI draft failed automatic checks, so it is not shown: " + "; ".join(res["problems"]) + ".")
        md.append("Read the original clauses below.")
    elif status == "insufficient":
        md.append("### Not enough verified information")
        md.append("I could not find sufficient verified information in the currently loaded documents to establish this requirement. "
                  + res.get("reason", ""))
        md.append("It may be covered by a document that has not been loaded. The closest clauses found are shown below only as a starting point, not as an answer.")
    elif status == "evidence_only":
        md.append("### Matching clauses")
        md.append("No AI model is connected, so there is no plain-language summary. The original clauses are shown below.")
    elif status == "llm_error":
        md.append("### AI explanation unavailable")
        md.append("The AI step failed (" + "; ".join(res["problems"]) + "). The original clauses are shown below.")

    md.append("")
    md.append("### Original regulation text (unaltered extraction)")
    if any(re.search(r"\bTable\s*(No\.?\s*)?\d", r.get("context", r["text"])) for r in res["results"]):
        md.append("*Some clauses below contain tables. PDF extraction flattens tables into plain text, so rows and "
                  "columns can be mixed up. Check the original table in the PDF before using any number.*")
    for i, r in enumerate(res["results"], start=1):
        src = source_info(r["source"], sources)
        end = r.get("page_end", r["page"])
        pages = f"page {r['page']}" if end == r["page"] else f"pages {r['page']}-{end}"
        where = f"`{r['source']}`, {pages}"
        if src:
            where = f"{src.get('title', r['source'])} · {src.get('authority', '')} · {pages}"
        md.append(f"**{i}. Clause {r['clause_no']}**  ·  {where}")
        full = r.get("context", r["text"])
        text = full[:1500] + ("…" if len(full) > 1500 else "")
        if r.get("n_passages", 1) > 1:
            text += f"  *(matching passages from this clause, joined; … marks a gap)*"
        md.append("> " + text.replace("\n", " "))
        if src:
            md.append(f"*Version/date: {src.get('version_note', 'not verified')}. Source: {src.get('url', 'not recorded')}*")
        md.append("")

    md.append(f"---\n*{DISCLAIMER}*")
    return "\n\n".join(m for m in md if m is not None)

In [ ]:
%%writefile eval.py
"""Measure retrieval quality honestly.   Usage:  python eval.py questions.csv [clauses.jsonl]

questions.csv columns:  question, correct_clause, qtype
  qtype is one of:  exact | everyday | mismatch | ambiguous | unanswerable
  correct_clause: the clause number you verified in the PDF. Leave it EMPTY for unanswerable questions
  (questions the loaded document cannot answer). A result is correct if its clause number equals the
  expected one or sits underneath it (expected 6.2 matches 6.2.1).

What it reports
  1. Retrieval on answerable questions: hit@5, hit@1, MRR for
        keyword | semantic | hybrid (raw question) | hybrid + term map (the app's real setting)
  2. hit@5 per question type, raw hybrid vs hybrid + term map (does understanding everyday wording help?)
  3. Refusal behaviour of the evidence check:
        answerable questions wrongly refused, unanswerable questions correctly refused
  4. Score diagnostics to help you recalibrate the thresholds in search.py

It measures RETRIEVAL only. Answer-writing quality is a separate test (see README).
Your test set is small and written by you, so treat results as indicative, not as a benchmark.
"""
import csv
import statistics
import sys
from collections import defaultdict

from search import MIN_OVERLAP, MIN_SEM, MIN_SEM_WITH_CONCEPT, Searcher, assess_evidence, load_clauses


def is_hit(clause_no, expected):
    return clause_no == expected or clause_no.startswith(expected + ".")


def rank_of(results, expected):
    for i, r in enumerate(results, start=1):
        if is_hit(r["clause_no"], expected):
            return i
    return None


def metrics(ranks):
    n = len(ranks) or 1
    return (sum(r is not None for r in ranks) / n,
            sum(r == 1 for r in ranks) / n,
            sum(1 / r for r in ranks if r) / n)


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else "questions.csv"
    clauses_path = sys.argv[2] if len(sys.argv) > 2 else "clauses.jsonl"
    with open(path, encoding="utf-8") as fh:
        rows = [r for r in csv.DictReader(fh) if (r.get("question") or "").strip()]
    for r in rows:
        r["qtype"] = (r.get("qtype") or "unlabeled").strip().lower()
        r["correct_clause"] = (r.get("correct_clause") or "").strip()
    answerable = [r for r in rows if r["correct_clause"] and r["qtype"] != "unanswerable"]
    unanswerable = [r for r in rows if r["qtype"] == "unanswerable" or not r["correct_clause"]]

    s = Searcher(load_clauses(clauses_path))
    print(f"Embedder: {s.embedder_name} | answerable: {len(answerable)} | unanswerable: {len(unanswerable)}\n")

    FINAL = "hybrid + term map + heading match"   # the app's real setting
    setups = [("keyword", "keyword", False, False), ("semantic", "semantic", False, False),
              ("hybrid (raw question)", "hybrid", False, False), ("hybrid + term map", "hybrid", True, False),
              (FINAL, "hybrid", True, True)]
    ranks = {name: [] for name, _, _, _ in setups}
    details = []
    for r in answerable:
        for name, mode, expand, titles in setups:
            rk = rank_of(s.search(r["question"], mode=mode, k=5, expand=expand, titles=titles), r["correct_clause"])
            ranks[name].append(rk)
            details.append((name, r["qtype"], r["question"], r["correct_clause"], rk))

    print("1. Retrieval on answerable questions (each clause counted once)")
    print(f"{'setup':<36}{'hit@5':>8}{'hit@1':>8}{'MRR':>8}")
    for name, _, _, _ in setups:
        h5, h1, mrr = metrics(ranks[name])
        print(f"{name:<36}{h5:>8.0%}{h1:>8.0%}{mrr:>8.2f}")

    print("\n2. hit@5 by question type (raw hybrid -> final setting)")
    by_type = defaultdict(list)
    for i, r in enumerate(answerable):
        by_type[r["qtype"]].append((ranks["hybrid (raw question)"][i], ranks[FINAL][i]))
    print(f"{'type':<14}{'n':>4}{'raw':>8}{'final':>8}")
    for t, pairs in sorted(by_type.items()):
        raw = sum(p[0] is not None for p in pairs) / len(pairs)
        mp = sum(p[1] is not None for p in pairs) / len(pairs)
        print(f"{t:<14}{len(pairs):>4}{raw:>8.0%}{mp:>8.0%}")

    print("\n3. Evidence check (refusal behaviour, before any AI step)")
    wrongly_refused, sem_ok, sem_bad = [], [], []
    for r in answerable:
        res = s.search(r["question"], k=5)
        info = s.analyze(r["question"])
        status, _ = assess_evidence(r["question"], info, res)
        sem_ok.append(max(x["sem_score"] for x in res))
        if status == "none":
            wrongly_refused.append(r["question"])
    correctly_refused, passed_through = [], []
    for r in unanswerable:
        res = s.search(r["question"], k=5)
        info = s.analyze(r["question"])
        status, _ = assess_evidence(r["question"], info, res)
        sem_bad.append(max(x["sem_score"] for x in res))
        (correctly_refused if status == "none" else passed_through).append(r["question"])
    print(f"answerable questions wrongly refused : {len(wrongly_refused)}/{len(answerable)}")
    for q in wrongly_refused:
        print("    -", q)
    if unanswerable:
        print(f"unanswerable questions refused       : {len(correctly_refused)}/{len(unanswerable)}")
        print("  (the ones that passed through must be caught by the AI step saying 'insufficient')")
        for q in passed_through:
            print("    passed through:", q)

    print("\n4. Score diagnostics (best semantic score per question)")
    if sem_ok:
        print(f"answerable   : min {min(sem_ok):.2f}  median {statistics.median(sem_ok):.2f}")
    if sem_bad:
        print(f"unanswerable : max {max(sem_bad):.2f}  median {statistics.median(sem_bad):.2f}")
    print(f"current thresholds: MIN_SEM={MIN_SEM}, MIN_SEM_WITH_CONCEPT={MIN_SEM_WITH_CONCEPT}, MIN_OVERLAP={MIN_OVERLAP}")
    print("If answerable questions are wrongly refused, lower the thresholds in search.py; if unanswerable ones pass, raise them.")

    with open("eval_details.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["setup", "qtype", "question", "expected_clause", "rank_of_correct (blank = not in top 5)"])
        w.writerows(details)
    print("\nPer-question results saved to eval_details.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile eval_answers.py
"""Measure the ANSWER step (separate from retrieval).  Needs a Gemini key:  GEMINI_API_KEY=... python eval_answers.py questions.csv

For each question it runs the whole pipeline and counts the outcomes:
  answerable questions   : answered | unverified (AI text withheld by checks) | insufficient (wrongly refused) | llm_error
                           plus: of the answered ones, how often a cited clause matches the clause you marked correct
  unanswerable questions : insufficient (correct refusal) | answered or unverified (the system tried to answer)

What this does NOT measure: whether the plain-language wording is clear or fully faithful. Read 10-15 answers
yourself and score them by hand (see README). Automatic checks catch invented numbers and fake citations only.
"""
import collections
import csv
import os
import sys
import time

from answer import answer_question, make_gemini_llm
from eval import is_hit
from search import Searcher, load_clauses


def run(searcher, rows, llm, delay=0.0):
    stats, details = collections.Counter(), []
    for r in rows:
        unans = r["qtype"] == "unanswerable" or not r["correct_clause"]
        res = answer_question(searcher, r["question"], llm=llm)
        group = "unanswerable" if unans else "answerable"
        stats[(group, res["status"])] += 1
        from answer import cited_clauses
        cited = cited_clauses(res["answer"] or {})
        cite_ok = None
        if not unans and res["status"] == "answered":
            cite_ok = any(is_hit(c, r["correct_clause"]) for c in cited)
            stats["answered_total"] += 1
            stats["cited_expected"] += int(cite_ok)
        details.append((r["qtype"], r["question"], r["correct_clause"], res["status"], ";".join(cited),
                        cite_ok, " | ".join(res["problems"])))
        if delay:
            time.sleep(delay)
    return stats, details


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else "questions.csv"
    clauses_path = sys.argv[2] if len(sys.argv) > 2 else "clauses.jsonl"
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        sys.exit("Set GEMINI_API_KEY first (see README).")
    llm = make_gemini_llm(key)
    with open(path, encoding="utf-8") as fh:
        rows = [r for r in csv.DictReader(fh) if (r.get("question") or "").strip()]
    for r in rows:
        r["qtype"] = (r.get("qtype") or "unlabeled").strip().lower()
        r["correct_clause"] = (r.get("correct_clause") or "").strip()
    searcher = Searcher(load_clauses(clauses_path))
    stats, details = run(searcher, rows, llm, delay=float(os.environ.get("CALL_DELAY", 4)))

    print(f"Model: {llm.model_name} | questions: {len(rows)}\n")
    for group in ("answerable", "unanswerable"):
        total = sum(v for k, v in stats.items() if isinstance(k, tuple) and k[0] == group)
        print(f"{group} questions ({total}):")
        for status in ("answered", "unverified", "insufficient", "llm_error", "evidence_only"):
            n = stats[(group, status)]
            if n:
                print(f"  {status:<14}{n:>4}  ({n / total:.0%})")
    if stats["answered_total"]:
        print(f"\nOf {stats['answered_total']} answered, a cited clause matched your expected clause in "
              f"{stats['cited_expected']} ({stats['cited_expected'] / stats['answered_total']:.0%}).")
    with open("answer_eval_details.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["qtype", "question", "expected", "status", "cited", "cited_expected", "problems"])
        w.writerows(details)
    print("Per-question results saved to answer_eval_details.csv. Now read 10-15 of the answers yourself.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile sources.json
[
  {
    "match": ["part_ii_planning", "part2", "part_2", "part-ii"],
    "title": "Comprehensive General Development Control Regulations 2017, Part II: Planning Regulation",
    "authority": "Urban Development and Urban Housing Department, Government of Gujarat",
    "url": "https://ahmedabadcity.gov.in/Uploads/FormsFonts/Town%20Development%20Department%20(B.P.S.P.)/Final_Notification_CGDCR_2017_PART_II_PLANNING_REGULATION_min.pdf",
    "version_note": "not verified. Check the document for amendments and the current status"
  }
]

## Step 3 (YOUR TURN): upload the PDF
Run the cell below, click **Choose files**, and pick the CGDCR 2017 Part II (Planning Regulation) PDF. The text in the PDF must be selectable.

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded:", pdf_name)

## Step 4: split the PDF into clauses
This keeps chapters 6-11, which hold the planning rules, and skips contents pages, page headers and table rows.

In [ ]:
!python ingest.py "$pdf_name" --out clauses.jsonl --chapters 6-11

### Check the result (YOUR TURN)
Read the lines below. Each line is one clause. Do the clause numbers look real (for example 6.7.1) and does the text look like actual rules? If it looks messy, the PDF may be scanned or oddly formatted; try another copy of the document.

In [ ]:
!head -n 12 clauses.jsonl

### Clause index (YOUR TURN)
This lists every distinct clause that was extracted, so we can see what the tool actually has. It also downloads a small file called `clause_index.txt` to your computer for reference.

In [ ]:
import json
index = {}
for line in open("clauses.jsonl", encoding="utf-8"):
    d = json.loads(line)
    index.setdefault((d["clause_no"], d["title"][:70]), d["page"])
lines = [f"{no} | {title} | p.{pg}" for (no, title), pg in index.items()]
open("clause_index.txt", "w", encoding="utf-8").write("\n".join(lines))
print(len(lines), "distinct clauses. First 40:")
print("\n".join(lines[:40]))
from google.colab import files
files.download("clause_index.txt")

## Step 5: try a search
This loads the search engine (the first run downloads a small language model) and tests one question.

In [ ]:
from search import Searcher, load_clauses, assess_evidence
searcher = Searcher(load_clauses("clauses.jsonl"))
print("Embedding method in use:", searcher.embedder_name)

question = "Can I build my house right up to the edge of my land?"
info = searcher.analyze(question)
print("Topics understood:", info["concepts"])
print("Extra terms searched:", info["terms"])
print("Details not given:", info["missing_details"])
results = searcher.search(question, mode="hybrid", k=5)
for r in results:
    print(r["clause_no"], "| page", r["page"], "| score", round(r["sem_score"], 2), "|", r["title"][:70])
print("Evidence check:", assess_evidence(question, info, results))

### Step 5b: where do the margin clauses rank? (YOUR TURN)
The clause index shows that 6.7 Margins, 6.7.1 Road side margin and 6.7.2 Rear and other side margins exist. This cell shows where each search setting ranks them for a margin question, and prints what the extracted text of 6.7.1 and 6.7.2 actually looks like. Check that the expected clauses rank near the top.

In [ ]:
q = "Can I build my house right up to the edge of my land?"
wanted = ["6.7", "6.7.1", "6.7.2", "6.7.3", "6.7.4"]
for mode, expand, titles in [("keyword", False, False), ("semantic", False, False),
                             ("hybrid", False, False), ("hybrid", True, False), ("hybrid", True, True)]:
    res = searcher.search(q, mode=mode, k=100, expand=expand, titles=titles)
    pos = {r["clause_no"]: i for i, r in enumerate(res, 1)}
    label = f"{mode}, term map={expand}, heading match={titles}"
    print(f"{label:<52}", {c: pos.get(c, ">100") for c in wanted})

print()
print("What the final setting returns:")
for r in searcher.search(q, mode="hybrid", k=5):
    print(" ", r["clause_no"], "| pages", r["page"], "-", r["page_end"], "|", r["title"][:60])

print()
for r in searcher.search("road side margin", mode="keyword", k=100, expand=False):
    if r["clause_no"] in ("6.7", "6.7.1", "6.7.2"):
        print("-----", r["clause_no"], "| pages", r["page"], "-", r["page_end"], "| passages:", r["n_passages"])
        print(r["context"][:700])
        print()

## Step 6: the test questions (already written for you)
Just run the next cell. It saves 45 questions: even rows for tuning, odd rows for the final test.

In [ ]:
# Step 6 (made for you): the question list. Just run this cell.
import csv, io
RAW = """question,correct_clause,qtype
What is the road side margin required for a building unit?,6.7.1,exact
What are the permissible uses in the margin space?,6.7.3,exact
What is the maximum clear height of a basement?,6.8.3,exact
Which uses are permitted in a basement?,6.8.4,exact
What is the minimum area of a building unit for different uses?,6.4,exact
What is the parking requirement for dwelling units?,6.15,exact
What is the minimum common plot required for a building unit of 2000 sq.mt or above?,6.17.5,exact
What are the requirements for fire tender movement?,6.14.3,exact
What is the minimum plot area for row type buildings?,6.13.1,exact
How close to my neighbour's boundary can I build?,6.7.2,everyday
Can I build a garage or servant room in the open space around my house?,6.7.3,everyday
Can I build a basement under my house?,6.8,everyday
How tall can my building be on a 12 metre road?,6.6,everyday
How much total floor area can I build on my plot?,6.3,everyday
Can I build on a plot that has no road access?,6.5.2,everyday
Do I need to leave open space inside my building for light and air?,6.16,everyday
Can I build on land marked as a graveyard?,6.18,everyday
How small can a plot be for building a house?,6.4,everyday
Can I add another floor to my existing house?,6.9,everyday
Can I run a small business from my house?,6.1,everyday
How many car parking spaces must a new apartment building provide?,6.15,everyday
What happens to my plot if the road in front is widened?,6.3.3,everyday
Can I put up a hoarding on my plot?,11.1,everyday
How much open spce do I need beteen my house and the boundry?,6.7,everyday
Parkng space needed for a shop,6.15,everyday
Can I build my house right up to the edge of my land?,6.7,mismatch
My plot is behind other houses and I cannot reach the road. What can I do?,6.5.2,mismatch
Is there a minimum gap I must keep between two buildings?,6.7.1,mismatch
I want to rebuild my 40 year old house on a narrow lane. What are the rules?,6.22,mismatch
Can I use the space below my building for storing things?,6.8.4,mismatch
I want to open a petrol pump. Where can it be located?,10,mismatch
Do tall buildings have to be reachable by fire engines?,6.14.3,mismatch
What can I build on a plot in the old walled city?,7,mismatch
Can I build on a plot reserved as a public garden under a town planning scheme?,6.19,mismatch
How close can I build to the road?,6.7.1,ambiguous
How high can I build?,6.6,ambiguous
How much can I build on my plot?,6.3,ambiguous
How much parking do I need?,6.15,ambiguous
What is the penalty for building without permission?,,unanswerable
What documents do I need to apply for building permission?,,unanswerable
How big should a septic tank be?,,unanswerable
What is the stamp duty on buying a plot?,,unanswerable
What is the income tax rate for salaried employees?,,unanswerable
Which cement grade is best for slabs?,,unanswerable
What is the minimum wage for construction workers?,,unanswerable
"""
rows = list(csv.DictReader(io.StringIO(RAW)))
def save(name, rs):
    with open(name, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["question","correct_clause","qtype"])
        for r in rs: w.writerow([r["question"], r["correct_clause"], r["qtype"]])
save("questions.csv", rows)
save("questions_tune.csv", rows[0::2])   # even rows: used for tuning
save("questions_test.csv", rows[1::2])   # odd rows: only for the final score
print(len(rows), "questions saved:", len(rows[0::2]), "tune /", len(rows[1::2]), "test")


## Step 7: measure retrieval quality
Compares keyword, semantic and hybrid search, with and without the everyday-term map, and checks refusals.

While you improve things, run it on `questions_tune.csv`. When you stop changing the code, run it **once** on `questions_test.csv` and report only those numbers, together with how many questions there are.

In [ ]:
!python eval.py questions_tune.csv clauses.jsonl

In [ ]:
!python eval.py questions_test.csv clauses.jsonl

In [ ]:
import os
os.environ["GEMINI_MODEL"] = "gemini-flash-lite-latest"
os.environ["CALL_DELAY"] = "6"


## Step 8 (YOUR TURN): connect the AI model for plain-language answers
1. Go to **aistudio.google.com**, sign in, and create an API key (check the page for current free-tier limits).
2. In Colab, click the **key icon** in the left sidebar (Secrets), add a secret named `GEMINI_API_KEY`, paste the key, and switch on **Notebook access**.
3. Run the cell below. Never paste the key into a cell or share it.

Without a key everything still works; you just get the original clauses and no plain-language summary.

In [ ]:
import os
from answer import make_gemini_llm
llm = None
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = make_gemini_llm(os.environ["GEMINI_API_KEY"])
    print("Connected. Model in use:", llm.model_name)
except Exception as e:
    print("No AI model connected:", e)
    print("The demo below will show original clauses only.")

## Step 9: try the full pipeline on one question

In [ ]:
from answer import answer_question
from render import to_markdown
from IPython.display import Markdown
res = answer_question(searcher, "Can I build my house right up to the edge of my land?", llm=llm)
print("Status:", res["status"], "| problems:", res["problems"])
Markdown(to_markdown(res))

## Step 10: measure the answer step
Runs your test questions through the whole pipeline and counts: answered, withheld by the checks, correctly refused, and whether the cited clause is the one you marked correct. It takes a few minutes because it pauses between calls.

These automatic checks only catch invented numbers and fake citations. **Also read 10-15 answers yourself** and judge clarity and faithfulness by hand.

In [ ]:
import os
os.environ["CALL_DELAY"] = "4"
!python eval_answers.py questions_test.csv clauses.jsonl

### Step 10b: the 20 everyday-wording questions
These are 20 questions written in everyday words (14 the document answers, 6 it does not). The expected clause numbers in the file are used **only for scoring after the run**. They are never given to the search or the model.
This is a development set, not a clean test: the prompt was changed after seeing these results. Expect different numbers between runs.

In [ ]:
%%writefile questions_user20.csv
question,correct_clause,qtype
How much space do I need to leave between my house and my neighbour's property?,6.7.2,everyday
Can I add another floor to my existing two-storey house?,6.9,everyday
How much of my total plot area am I allowed to cover with construction?,6.3.1,everyday
Can I build a balcony that extends over the open space around my house?,6.7.3,everyday
What is the minimum road width required to construct a residential building on my plot?,6.2,everyday
Can I build a house on a plot that is smaller than the usual residential plot size?,6.4,everyday
"How tall can I make my house on a 1,000-square-foot plot?",6.6,everyday
"Do I need to leave open space at the back of my house, or is space on the sides enough?",6.7.2,everyday
Can I construct a small room on the terrace of my existing house?,,unanswerable
"If my neighbour has built very close to our shared boundary, can I do the same?",6.7.2,everyday
How many parking spaces do I need to provide when building a house?,6.15,everyday
Can I convert the ground floor of my house into a small shop?,6.2,everyday
Is it allowed to build a house on a plot that faces a narrow road?,6.2,everyday
Can I extend my kitchen or bedroom into the open space beside my house?,6.7.4,everyday
Do I need permission before making changes to an already constructed house?,,unanswerable
Can I demolish part of my existing house and rebuild it with an additional floor?,6.9,everyday
Are there different construction rules for corner plots?,,unanswerable
Can I build a compound wall right along the edge of my property?,,unanswerable
What happens if the house I bought was constructed without following the approved building plan?,,unanswerable
"If building rules have changed recently, do I have to follow the new rules when extending my old house?",,unanswerable

In [ ]:
!python eval_answers.py questions_user20.csv clauses.jsonl

## Step 11: launch a demo you can share
Run the cell below. It prints a public link that works while this notebook stays running and expires later.

In [ ]:
import gradio as gr
from answer import answer_question
from render import to_markdown

def ask(question):
    if not question.strip():
        return "Type a question first."
    return to_markdown(answer_question(searcher, question, llm=llm))

demo = gr.Interface(
    fn=ask,
    inputs=gr.Textbox(label="Your question", lines=2, placeholder="e.g. Can I build my house right up to the edge of my land?"),
    outputs=gr.Markdown(label="Answer and evidence"),
    title="Clause finder (prototype)",
    description="Ask in everyday language. You get a plain-language answer built only from the clauses shown, with the original text, page and source. Research aid only, not legal or engineering advice.",
)
demo.launch(share=True)